# Compare machine-learning models for CRISPR activity prediction

**Task chosen:** regression of measured guide on-target activity from guide sequence. This is not an off-target classifier, screen-hit ranking or indel-outcome model.

Models: **Mean baseline, Ridge regression, SVR, Random Forest, XGBoost**. Tools from the brochure: NumPy, Pandas, scikit-learn, XGBoost, Matplotlib and Seaborn. SciPy supplies rank correlation; joblib exports the selected model.

The notebook includes a synthetic demonstration and a real-data upload route. **Synthetic performance is not biological validation.** No API key or GPU is required.


## 1. Install packages
Use a CPU runtime. Installation requires internet. All five models are included by default; missing XGBoost raises an error rather than silently omitting it.

In [ ]:
from pathlib import Path
import sys, subprocess, json, shutil
from datetime import datetime
ROOT=Path.cwd()/'crispr_benchmark_work'
ROOT.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','numpy>=1.24,<3','pandas>=2.0,<3','scipy>=1.11,<2','scikit-learn>=1.4,<2','xgboost>=2.1,<4','matplotlib>=3.7,<4','seaborn>=0.13,<0.14','joblib>=1.3,<2'],check=True)


## 2. Write the complete shared benchmark program
The program is also supplied separately for Ubuntu. Models run in a fresh Python subprocess after installation.

In [ ]:
(ROOT/"benchmark.py").write_text('#!/usr/bin/env python3\n"""Group-aware regression benchmark for measured CRISPR guide activity."""\nimport argparse, hashlib, json, time, platform, warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import spearmanr\nfrom sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV, cross_val_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.dummy import DummyRegressor\nfrom sklearn.linear_model import Ridge\nfrom sklearn.svm import SVR\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\nimport joblib\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n\ndef features(sequences):\n    bases=\'ACGT\'; L=len(sequences[0]); rows=[]\n    names=[f\'pos_{i+1}_{b}\' for i in range(L) for b in bases]+[\'GC_fraction\',\'max_homopolymer\',\'base_entropy\']+[a+b+\'_fraction\' for a in bases for b in bases]\n    for s in sequences:\n        f=[float(c==b) for c in s for b in bases]\n        prob=np.array([s.count(b)/L for b in bases]); nz=prob[prob>0]\n        longest=cur=1\n        for i in range(1,L):\n            cur=cur+1 if s[i]==s[i-1] else 1; longest=max(longest,cur)\n        f += [(s.count(\'G\')+s.count(\'C\'))/L,longest,float(-(nz*np.log2(nz)).sum())]\n        f += [sum(s[i:i+2]==a+b for i in range(L-1))/(L-1) for a in bases for b in bases]\n        rows.append(f)\n    return np.asarray(rows),names\n\n\ndef validate(df,length):\n    needed=[\'guide_id\',\'guide_sequence\',\'activity\',\'target_group\']\n    if not set(needed)<=set(df): raise ValueError(\'Required CSV columns: \'+\',\'.join(needed))\n    x=df[needed].copy()\n    if x.isna().any().any(): raise ValueError(\'Missing values are not allowed.\')\n    for c in [\'guide_id\',\'target_group\',\'guide_sequence\']:\n        x[c]=x[c].astype(str).str.strip()\n        if x[c].eq(\'\').any(): raise ValueError(\'Empty value in \'+c)\n    x.guide_sequence=x.guide_sequence.str.upper()\n    if not x.guide_sequence.str.fullmatch(\'[ACGT]{\'+str(length)+\'}\').all():\n        raise ValueError(f\'Use exactly {length} A/C/G/T guide bases, excluding PAM. No N bases or mixed lengths.\')\n    if x.guide_id.duplicated().any(): raise ValueError(\'guide_id must be unique.\')\n    if x.guide_sequence.duplicated().any():\n        raise ValueError(\'Repeated guide sequence detected. Resolve duplicate records/replicates within a consistent assay before benchmarking. Do not average incompatible conditions.\')\n    x.activity=pd.to_numeric(x.activity,errors=\'raise\')\n    if not np.isfinite(x.activity).all() or x.activity.nunique()<3:\n        raise ValueError(\'activity must be finite numeric measurements with meaningful variation, not binary classes.\')\n    if len(x)<40 or x.target_group.nunique()<12:\n        raise ValueError(\'This benchmark requires at least 40 unique guides and 12 independent target groups; more data is preferable.\')\n    return x.reset_index(drop=True)\n\n\ndef demo(path):\n    rng=np.random.default_rng(42); rows=[]\n    for g in range(30):\n        group_effect=rng.normal(0,.07)\n        for j in range(12):\n            seq=\'\'.join(rng.choice(list(\'ACGT\'),20)); gc=(seq.count(\'G\')+seq.count(\'C\'))/20\n            y=.45+.25*(seq[3]==\'G\')-.18*(seq[15]==\'T\')-.6*(gc-.5)**2+group_effect+rng.normal(0,.05)\n            rows.append([f\'demo_{g:02d}_{j:02d}\',seq,float(np.clip(y,0,1)),f\'DEMO_TARGET_{g:02d}\'])\n    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)\n    if p.exists(): raise FileExistsError(p)\n    pd.DataFrame(rows,columns=[\'guide_id\',\'guide_sequence\',\'activity\',\'target_group\']).to_csv(p,index=False)\n    print(\'SIMULATED teaching data:\',p)\n\n\ndef models(seed,quick,names):\n    trees=60 if quick else 300\n    specs={\n      \'MeanBaseline\':(DummyRegressor(strategy=\'mean\'),{}),\n      \'Ridge\':(Pipeline([(\'scale\',StandardScaler()),(\'model\',Ridge())]),{\'model__alpha\':[.1,1,10,100,1000]}),\n      \'SVR\':(Pipeline([(\'scale\',StandardScaler()),(\'model\',SVR())]),{\'model__C\':[.1,1,10,100],\'model__epsilon\':[.01,.05,.1],\'model__gamma\':[\'scale\',.001,.01]}),\n      \'RandomForest\':(RandomForestRegressor(n_estimators=trees,random_state=seed,n_jobs=1),{\'max_depth\':[None,5,10],\'min_samples_leaf\':[1,3,6],\'max_features\':[.5,1.0]})\n    }\n    if \'XGBoost\' in names:\n        try: from xgboost import XGBRegressor\n        except ImportError as e: raise ImportError(\'Install requirements.txt to include XGBoost; or explicitly choose --models MeanBaseline,Ridge,SVR,RandomForest.\') from e\n        specs[\'XGBoost\']=(XGBRegressor(objective=\'reg:squarederror\',tree_method=\'hist\',n_estimators=trees,random_state=seed,n_jobs=1),\n                          {\'max_depth\':[2,3,5],\'learning_rate\':[.03,.1,.2],\'subsample\':[.8,1.0],\'colsample_bytree\':[.8,1.0],\'reg_lambda\':[1,5]})\n    bad=set(names)-set(specs)\n    if bad: raise ValueError(\'Unknown models: \'+str(bad))\n    return {k:specs[k] for k in names}\n\n\ndef metrics(y,p):\n    rho=float(spearmanr(y,p).statistic) if np.std(y)>0 and np.std(p)>0 else np.nan\n    r2=float(r2_score(y,p)) if np.std(y)>0 else np.nan\n    return {\'RMSE\':float(np.sqrt(mean_squared_error(y,p))),\'MAE\':float(mean_absolute_error(y,p)),\'R2\':r2,\'Spearman\':rho}\n\n\ndef run(a):\n    if not .1<=a.test_fraction<=.4 or a.bootstrap<100 or a.sequence_length<2: raise ValueError(\'Invalid settings.\')\n    df=validate(pd.read_csv(a.input,dtype={\'guide_id\':str,\'guide_sequence\':str,\'target_group\':str}),a.sequence_length)\n    names=list(dict.fromkeys(a.models.split(\',\')))\n    specs=models(a.seed,a.quick,names)\n    X,feature_names=features(df.guide_sequence.tolist()); y=df.activity.to_numpy(); groups=df.target_group.to_numpy()\n    split=GroupShuffleSplit(n_splits=1,test_size=a.test_fraction,random_state=a.seed)\n    train,test=next(split.split(X,y,groups))\n    if len(set(groups[test]))<3 or len(set(groups[train]))<5:\n        raise ValueError(\'Need >=3 held-out groups and >=5 training groups. Increase groups/test_fraction.\')\n    assert not set(groups[train]) & set(groups[test])\n    assert not set(df.guide_sequence.iloc[train]) & set(df.guide_sequence.iloc[test])\n    folds=min(3 if a.quick else 5,len(set(groups[train])))\n    # Materialize identical group-aware folds for every candidate model.\n    cv=list(GroupKFold(n_splits=folds).split(X[train],y[train],groups[train]))\n    out=Path(a.out); out.mkdir(parents=True,exist_ok=False)\n    split_table=df[[\'guide_id\',\'target_group\']].copy(); split_table[\'split\']=\'train\'; split_table.loc[test,\'split\']=\'test\'\n    split_table[\'cv_validation_fold\']=pd.Series([None]*len(df),dtype=\'Int64\')\n    for f,(_,valid) in enumerate(cv): split_table.loc[train[valid],\'cv_validation_fold\']=f\n    split_table.to_csv(out/\'split_manifest.csv\',index=False)\n    fitted={}; cv_rows=[]; best_parameters={}\n    # All tuning and model-family selection take place before test predictions.\n    for name,(estimator,space) in specs.items():\n        start=time.perf_counter()\n        if space:\n            search=RandomizedSearchCV(estimator,space,n_iter=2 if a.quick else 10,scoring=\'neg_root_mean_squared_error\',cv=cv,random_state=a.seed,n_jobs=a.jobs,refit=True,error_score=\'raise\')\n            search.fit(X[train],y[train])\n            fitted[name]=search.best_estimator_; cv_rmse=-search.best_score_; params=search.best_params_\n            trials=pd.DataFrame(search.cv_results_); trials.to_csv(out/f\'{name}_tuning.csv\',index=False)\n        else:\n            cv_rmse=-cross_val_score(estimator,X[train],y[train],cv=cv,scoring=\'neg_root_mean_squared_error\',n_jobs=a.jobs,error_score=\'raise\').mean()\n            estimator.fit(X[train],y[train]); fitted[name]=estimator; params={}\n        cv_rows.append({\'model\':name,\'training_CV_RMSE\':cv_rmse,\'fit_and_search_seconds\':time.perf_counter()-start})\n        best_parameters[name]=params\n        print(\'Fitted:\',name,flush=True)\n    selected=min(cv_rows,key=lambda row:row[\'training_CV_RMSE\'])[\'model\']\n    pd.DataFrame(cv_rows).sort_values(\'training_CV_RMSE\').to_csv(out/\'training_cv_comparison.csv\',index=False)\n    preds={name:est.predict(X[test]) for name,est in fitted.items()}\n    pred_table=df.iloc[test].copy()\n    for name,pred in preds.items(): pred_table[name+\'_prediction\']=pred\n    pred_table.to_csv(out/\'heldout_predictions.csv\',index=False)\n    # Cluster bootstrap: resample whole test groups, same samples for all models.\n    rng=np.random.default_rng(a.seed+1); ug=np.unique(groups[test]); indices={g:np.flatnonzero(groups[test]==g) for g in ug}\n    boot={name:[] for name in names}\n    for _ in range(a.bootstrap):\n        pick=np.concatenate([indices[g] for g in rng.choice(ug,size=len(ug),replace=True)])\n        for name,pred in preds.items(): boot[name].append(metrics(y[test][pick],pred[pick]))\n    rows=[]\n    for name,pred in preds.items():\n        row={\'model\':name,\'selected_by_training_CV\':name==selected,**metrics(y[test],pred)}\n        for m in [\'RMSE\',\'MAE\',\'R2\',\'Spearman\']:\n            values=np.array([b[m] for b in boot[name]]); values=values[np.isfinite(values)]\n            lo,hi=np.quantile(values,[.025,.975]) if len(values) else (np.nan,np.nan)\n            row[m+\'_CI_low\']=lo; row[m+\'_CI_high\']=hi\n        rows.append(row)\n    result=pd.DataFrame(rows)\n    # Keep CV order: the test set is for reporting, not further model selection.\n    result=result.set_index(\'model\').loc[pd.DataFrame(cv_rows).sort_values(\'training_CV_RMSE\').model].reset_index()\n    result.to_csv(out/\'heldout_metrics.csv\',index=False)\n    if \'MeanBaseline\' in names:\n        improvements=[]\n        for name in names:\n            if name==\'MeanBaseline\': continue\n            diffs=np.array([b[\'RMSE\']-m[\'RMSE\'] for b,m in zip(boot[\'MeanBaseline\'],boot[name])])\n            improvements.append({\'model\':name,\'mean_baseline_RMSE_minus_model_RMSE\':metrics(y[test],preds[\'MeanBaseline\'])[\'RMSE\']-metrics(y[test],preds[name])[\'RMSE\'],\n                                 \'paired_group_bootstrap_low\':np.quantile(diffs,.025),\'paired_group_bootstrap_high\':np.quantile(diffs,.975)})\n        pd.DataFrame(improvements).to_csv(out/\'improvement_over_baseline.csv\',index=False)\n    sns.set_theme(style=\'whitegrid\')\n    fig,ax=plt.subplots(figsize=(8,4)); xp=np.arange(len(result))\n    ax.bar(xp,result.RMSE,color=\'#458bb7\'); ax.vlines(xp,result.RMSE_CI_low,result.RMSE_CI_high,color=\'black\',lw=2)\n    ax.set_xticks(xp,result.model,rotation=20); ax.set(ylabel=\'Held-out RMSE (lower is better)\',title=(\'SIMULATED | \' if a.demo_label else \'\')+\'Held-out target groups\')\n    fig.tight_layout(); fig.savefig(out/\'model_comparison.png\',dpi=160); plt.close(fig)\n    fig,axs=plt.subplots(1,len(names),figsize=(4*len(names),4),squeeze=False)\n    for ax,(name,pred) in zip(axs[0],preds.items()):\n        ax.scatter(y[test],pred,s=15,alpha=.65); lo=min(y[test].min(),pred.min()); hi=max(y[test].max(),pred.max()); ax.plot([lo,hi],[lo,hi],\'k--\')\n        ax.set(title=name,xlabel=\'Observed activity\',ylabel=\'Predicted activity\')\n    fig.tight_layout(); fig.savefig(out/\'predicted_vs_observed.png\',dpi=160); plt.close(fig)\n    joblib.dump({\'model\':fitted[selected],\'feature_names\':feature_names,\'sequence_length\':a.sequence_length,\'selection\':\'training grouped CV only\',\'name\':selected},out/\'selected_model.joblib\')\n    report=vars(a).copy(); report.update({\'selected_model\':selected,\'best_parameters\':best_parameters,\'training_groups\':len(set(groups[train])),\'test_groups\':len(ug),\'training_rows\':len(train),\'test_rows\':len(test),\'CV_folds\':folds,\'python\':platform.python_version(),\'mode\':\'SIMULATED\' if a.demo_label else \'USER_DATA\',\'input_sha256\':hashlib.sha256(Path(a.input).read_bytes()).hexdigest(),\'features\':feature_names,\'primary_metric\':\'Row-weighted RMSE\',\'interval_scope\':\'95% cluster bootstrap intervals conditional on fitted models and observed held-out groups; not retraining uncertainty\'})\n    (out/\'run_manifest.json\').write_text(json.dumps(report,indent=2))\n    print(\'Selected by training CV:\',selected,\'| Results:\',out)\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--make-demo\'); p.add_argument(\'--input\'); p.add_argument(\'--out\',default=\'benchmark_results\')\n    p.add_argument(\'--models\',default=\'MeanBaseline,Ridge,SVR,RandomForest,XGBoost\')\n    p.add_argument(\'--sequence-length\',type=int,default=20); p.add_argument(\'--test-fraction\',type=float,default=.2)\n    p.add_argument(\'--bootstrap\',type=int,default=1000); p.add_argument(\'--jobs\',type=int,default=2); p.add_argument(\'--seed\',type=int,default=42)\n    p.add_argument(\'--quick\',action=\'store_true\'); p.add_argument(\'--demo-label\',action=\'store_true\')\n    a=p.parse_args()\n    if a.make_demo: demo(a.make_demo)\n    elif a.input: run(a)\n    else: p.error(\'Provide --input or --make-demo\')\nif __name__==\'__main__\': main()\n')


## Required input
`activity.csv` must contain:

| Column | Meaning |
|---|---|
| guide_id | Unique identifier for each guide |
| guide_sequence | 20 A/C/G/T bases, spacer only, excluding PAM; consistent 5′→3′ convention |
| activity | Measured numeric activity, with the same assay definition, units and normalization across records |
| target_group | Gene/locus or a more conservative cluster used to prevent related guides crossing splits |

Use experimentally measured labels. Guide-design scores predicted by another model are not experimental truth. Growth/fitness screen counts are not automatically direct editing activity. Do not pool incompatible assays, cell contexts, nucleases or label scales without an explicit harmonization design.

The default is 20-base spacers; use a fixed alternative length only with a compatible study and the matching sequence-length option. N bases and mixed lengths are rejected. This benchmark is sequence-only: it cannot account for cell type, chromatin or assay effects not represented in the features.

At least 40 unique guides and 12 independent groups are required by this workflow, with at least three held-out and five training groups. These are software minimums, not guarantees of statistical power. More groups and experimental validation are preferable. Repeated sequence records are rejected: resolve exact duplicates and technical/biological replicate labels within a coherent assay before input. Do not average incompatible conditions.

Choose groups before examining performance. Use the broadest relevant dependency unit: if near-identical sequences, related loci or study batches can cause leakage, assign them to the same appropriate cluster. This code blocks exact duplicate guides and group overlap, but does not automatically cluster sequence homology or detect all batch confounding.


## 3. Choose demonstration or upload measured activities

In [ ]:
DEMO_MODE=True
INPUT=ROOT/'input'
INPUT.mkdir(exist_ok=True)
if DEMO_MODE:
    DATA=INPUT/('demo_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.csv')
    subprocess.run([sys.executable,str(ROOT/'benchmark.py'),'--make-demo',str(DATA)],check=True)
else:
    try:
        from google.colab import files
    except ImportError:
        print('Copy activity.csv to',INPUT)
    else:
        for name,data in files.upload().items():
            (INPUT/Path(name).name).write_bytes(data)
    DATA=INPUT/'activity.csv'
    if not DATA.exists(): raise FileNotFoundError(DATA)
print('SIMULATED DATA' if DEMO_MODE else 'USER DATA',DATA)


## Benchmark design
1. Encode each sequence into positional one-hot features, GC fraction, longest homopolymer, base entropy and overlapping dinucleotide frequencies. For a 20-base guide there are 99 features. These features require no fitted data transformations.
2. Hold out 20% of groups using a fixed seed; this is a group fraction, not an exact row fraction.
3. Tune only training data with identical GroupKFold splits for all models. Scaling for Ridge and SVR is fitted inside each fold using a Pipeline.
4. Choose the model family and parameters using training-CV RMSE only. No early stopping uses the test set.
5. Evaluate all fitted models once on held-out groups. Save the training-CV-selected model fitted only on the training partition.
6. Resample whole held-out groups for 95% cluster-bootstrap intervals. All models use the same resampled groups, enabling paired comparison to the baseline.

Primary metric is **row-weighted RMSE** (lower is better); groups with more guides contribute more rows. MAE is also lower-is-better; R² and Spearman are higher-is-better. R² can be negative. Constant predictions have undefined Spearman; the mean baseline therefore has a missing Spearman value rather than a misleading zero. Metrics depend on label scale.

The best tuning-CV score is used for selection and can be optimistic; it is not an independent generalization estimate. This workflow uses one held-out split, not repeated nested CV or external validation. Intervals are conditional on the trained models and observed test groups, exclude training/tuning uncertainty, and can be unstable with few groups. Paired baseline intervals are descriptive and are not adjusted for multiple model comparisons.

Do not change features, seed or hyperparameters in response to the test results and continue calling that same set an untouched test. An external dataset with comparable assay labels is the next evaluation step.


## 4. Run the comparison
Full mode uses five grouped CV folds and up to ten parameter combinations per model (Ridge has five possible values). Quick mode uses three folds, two combinations and smaller tree ensembles. CPU runtime depends on dataset size. Select settings before examining the holdout.

In [ ]:
QUICK_MODE=True  # use False for the full training-only hyperparameter search
MODELS='MeanBaseline,Ridge,SVR,RandomForest,XGBoost'
SEQUENCE_LENGTH=20
TEST_GROUP_FRACTION=0.20
BOOTSTRAP_REPEATS=1000
SEED=42
JOBS=2
RESULTS=ROOT/(('demo_' if DEMO_MODE else 'real_')+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
cmd=[sys.executable,str(ROOT/'benchmark.py'),'--input',str(DATA),'--out',str(RESULTS),
     '--models',MODELS,'--sequence-length',str(SEQUENCE_LENGTH),'--test-fraction',str(TEST_GROUP_FRACTION),
     '--bootstrap',str(BOOTSTRAP_REPEATS),'--seed',str(SEED),'--jobs',str(JOBS)]
if DEMO_MODE: cmd += ['--demo-label']
if QUICK_MODE: cmd += ['--quick']
subprocess.run(cmd,check=True)
with (RESULTS/'installed_versions.txt').open('w') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)


## 5. Compare performance
The marked selected model was chosen by training CV. Do not select another winner from test scores without obtaining a new independent evaluation set. Training/search times are measured locally and are not hardware-independent comparisons.

In [ ]:
from IPython.display import display, Image
import pandas as pd
cv=pd.read_csv(RESULTS/'training_cv_comparison.csv')
metrics=pd.read_csv(RESULTS/'heldout_metrics.csv')
display(cv)
display(metrics)
if (RESULTS/'improvement_over_baseline.csv').exists():
    display(pd.read_csv(RESULTS/'improvement_over_baseline.csv'))
for filename in ['model_comparison.png','predicted_vs_observed.png']:
    display(Image(filename=str(RESULTS/filename)))
manifest=json.loads((RESULTS/'run_manifest.json').read_text())
print('Selected by training CV:',manifest['selected_model'])
print('Independent groups: train=',manifest['training_groups'],'test=',manifest['test_groups'])


## 6. Check split integrity and results
- Confirm target groups represent the real dependencies in your data; gene IDs alone may not resolve cross-study or sequence-homology leakage.
- Check how many independent groups remain in the test set. Many guides from very few genes are not broad validation.
- RMSE/MAE are in your activity units. R² can be negative, and constant baseline predictions have undefined rank correlation.
- The confidence intervals resample test groups only; they do not reflect retraining variation.
- The current feature set omits chromatin, cell state and nuclease context. External generalization cannot be assumed.
- Sequence-only predictions must not be interpreted as experimentally validated guide performance.


In [ ]:
split=pd.read_csv(RESULTS/'split_manifest.csv')
train_groups=set(split.loc[split['split']=='train','target_group'])
test_groups=set(split.loc[split['split']=='test','target_group'])
assert train_groups.isdisjoint(test_groups)
print('Confirmed: no target-group overlap between train and test.')
display(pd.read_csv(RESULTS/'heldout_predictions.csv').head(10))


## 7. Download everything
The archive includes metrics, plots, prediction tables, split assignments, model, tuning results and provenance. Only load trusted joblib files.

In [ ]:
archive=shutil.make_archive(str(RESULTS),'zip',root_dir=RESULTS)
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(archive))
else:
    files.download(archive)


## References
[GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) · [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) · [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) · [XGBoost](https://xgboost.readthedocs.io/en/stable/python/python_api.html)
